# 🏧 Reporting Parc GAB — 2025 & 2026**Objectif** : Analyser les retraits effectués sur **l'ensemble du parc d'automates** pour identifier quelles banques confrères les utilisent le plus, GAB par GAB et au global (nos porteurs vs confrères).> 💡 **Notre banque = code `20041`** — tout autre code est considéré comme **confrère****Périmètre** : tous les `num_automate`, années 2025 + 2026 (colonne `moisannee` au format `MMAAAA`, ex: `102025` = octobre 2025)

## 0. Import des librairies

In [ ]:
import dataikuimport pandas as pdimport plotly.express as pximport plotly.graph_objects as gofrom plotly.subplots import make_subplots

## 1. Chargement du dataset

In [ ]:
# ⚠️ Remplacez par le nom exact de votre dataset de sortie dans Dataikudataset = dataiku.Dataset("NOM_DE_VOTRE_DATASET")df = dataset.get_dataframe()print(f"✅ Dataset chargé : {df.shape[0]:,} lignes, {df.shape[1]} colonnes")print(f"🏧 Nb automates distincts : {df['num_automate'].nunique()}")df.head()

## 2. Préparation des données### 2.1 Typage des colonnes numériques

In [ ]:
# --- Typage ---df['nb_retrait']    = pd.to_numeric(df['nb_retrait'],    errors='coerce').fillna(0)df['montant_total'] = pd.to_numeric(df['montant_total'], errors='coerce').fillna(0)df['moisannee']     = df['moisannee'].astype(str).str.strip()df['num_automate']  = df['num_automate'].astype(str).str.strip()print(f"Lignes avant nettoyage : {len(df):,}")print(f"Valeurs moisannee distinctes : {sorted(df['moisannee'].unique())}")

### 2.2 Parsing robuste de `moisannee` (format MMAAAA)⚠️ Attention : `moisannee` n'est **pas toujours sur 6 caractères**. Un mois à un chiffre (janvier à septembre)donne un code à **5 caractères** (ex: `12025` = janvier 2025), alors qu'un mois à deux chiffres(octobre à décembre) donne **6 caractères** (ex: `102025` = octobre 2025).On ne peut donc pas couper bêtement "les 4 derniers caractères = année" : il faut détecter la longueur.

In [ ]:
def parse_moisannee(x):    """Parse un code moisannee MMAAAA (5 ou 6 caractères) -> (annee:int, mois:int, periode_tri:str, periode_label:str)"""    x = str(x).strip()    if len(x) == 6:        mois, annee = int(x[:2]), int(x[2:])    elif len(x) == 5:        mois, annee = int(x[:1]), int(x[1:])    else:        return pd.Series({'annee': None, 'mois': None, 'periode_tri': None, 'periode_label': None})    periode_tri   = f"{annee:04d}-{mois:02d}"          # ex: "2025-10"  -> trie correctement    mois_labels   = ['', 'Jan', 'Fév', 'Mar', 'Avr', 'Mai', 'Jun',                      'Jul', 'Aoû', 'Sep', 'Oct', 'Nov', 'Déc']    periode_label = f"{mois_labels[mois]} {annee}"      # ex: "Oct 2025" -> lisible sur les graphes    return pd.Series({'annee': annee, 'mois': mois, 'periode_tri': periode_tri, 'periode_label': periode_label})df[['annee', 'mois', 'periode_tri', 'periode_label']] = df['moisannee'].apply(parse_moisannee)# Vérification : aucune ligne ne doit avoir échoué au parsingnb_echecs = df['periode_tri'].isna().sum()if nb_echecs > 0:    print(f"⚠️ {nb_echecs} lignes n'ont pas pu être parsées (moisannee invalide) — à vérifier !")    display(df[df['periode_tri'].isna()][['moisannee']].drop_duplicates())else:    print("✅ Toutes les lignes ont été parsées correctement.")print(f"\nPériode couverte : {df['periode_tri'].min()} → {df['periode_tri'].max()}")print(f"Années présentes : {sorted(df['annee'].dropna().unique())}")

### 2.3 Colonnes nos porteurs / confrères

In [ ]:
NOTRE_BANQUE = '20041'df['banque_affectation'] = df['banque_affectation'].astype(str).str.strip()df['type_porteur'] = df['banque_affectation'].apply(    lambda x: 'Nos porteurs' if x == NOTRE_BANQUE else 'Confrères')# Label affiché : code + libellédf['label_banque'] = df['banque_affectation'] + ' - ' + df['lib_banque_affectation'].fillna('Inconnu')print(f"Nos porteurs  : {df[df['type_porteur']=='Nos porteurs']['nb_retrait'].sum():,.0f} retraits")print(f"Confrères     : {df[df['type_porteur']=='Confrères']['nb_retrait'].sum():,.0f} retraits")

## 3. KPIs — Vue globale parc 2025-2026

In [ ]:
total_retraits = df['nb_retrait'].sum()total_montant  = df['montant_total'].sum()nb_banques     = df['banque_affectation'].nunique()nb_gab         = df['num_automate'].nunique()pct_nos        = df[df['type_porteur']=='Nos porteurs']['nb_retrait'].sum() / total_retraits * 100pct_confreres  = 100 - pct_nosprint("=" * 55)print(f"  🏧 Nb automates (GAB) : {nb_gab}")print(f"  📅 Période            : {df['periode_label'].iloc[df['periode_tri'].argmin()]} → {df['periode_label'].iloc[df['periode_tri'].argmax()]}")print(f"  🔢 Total retraits     : {total_retraits:,.0f}")print(f"  💶 Total montant      : {total_montant:,.0f} €")print(f"  🏦 Nb banques         : {nb_banques}")print(f"  ✅ Nos porteurs       : {pct_nos:.1f}%")print(f"  🤝 Confrères          : {pct_confreres:.1f}%")print("=" * 55)

## 4. Graphe 1 — Nos porteurs vs Confrères (parc entier)Vue d'ensemble : qui retire de l'argent sur nos GAB, au global.

In [ ]:
agg_type = df.groupby('type_porteur').agg(    nb_retrait    = ('nb_retrait',    'sum'),    montant_total = ('montant_total', 'sum')).reset_index()fig1 = px.pie(    agg_type,    names  = 'type_porteur',    values = 'nb_retrait',    title  = '🏧 Parc GAB — Répartition Nos Porteurs vs Confrères (Nb retraits, 2025-2026)',    color  = 'type_porteur',    color_discrete_map = {        'Nos porteurs' : '#003f88',        'Confrères'    : '#e85d04'    },    hole = 0.4)fig1.update_traces(textposition='inside', textinfo='percent+label')fig1.show()

## 5. Graphe 2 — Top 10 banques confrères (parc entier)Quelles banques concurrentes utilisent le plus nos automates ?

In [ ]:
top10_banques = (    df.groupby(['label_banque', 'type_porteur'])      .agg(nb_retrait=('nb_retrait', 'sum'))      .reset_index()      .sort_values('nb_retrait', ascending=False)      .head(10))fig2 = px.bar(    top10_banques,    x      = 'nb_retrait',    y      = 'label_banque',    color  = 'type_porteur',    orientation = 'h',    title  = '🏧 Parc GAB — Top 10 banques par nombre de retraits (2025-2026)',    labels = {'nb_retrait': 'Nb retraits', 'label_banque': 'Banque'},    color_discrete_map = {        'Nos porteurs' : '#003f88',        'Confrères'    : '#e85d04'    },    text   = 'nb_retrait')fig2.update_traces(texttemplate='%{text:,.0f}', textposition='outside')fig2.update_layout(yaxis={'categoryorder': 'total ascending'})fig2.show()

## 6. Graphe 3 — Évolution mensuelle 2025-2026 (Nos porteurs vs Confrères)Tri chronologique correct grâce à `periode_tri` (sinon "12025" se classerait après "92025" en tri texte).

In [ ]:
evol = (    df.groupby(['periode_tri', 'periode_label', 'type_porteur'])      .agg(nb_retrait=('nb_retrait', 'sum'))      .reset_index()      .sort_values('periode_tri'))# Ordre chronologique forcé pour l'axe X (au cas où Plotly trierait alphabétiquement)ordre_periodes = evol.sort_values('periode_tri')['periode_label'].drop_duplicates().tolist()fig3 = px.line(    evol,    x      = 'periode_label',    y      = 'nb_retrait',    color  = 'type_porteur',    markers= True,    title  = '🏧 Parc GAB — Évolution mensuelle des retraits (2025-2026)',    labels = {'nb_retrait': 'Nb retraits', 'periode_label': 'Mois'},    color_discrete_map = {        'Nos porteurs' : '#003f88',        'Confrères'    : '#e85d04'    },    category_orders = {'periode_label': ordre_periodes})fig3.update_xaxes(tickangle=-45)fig3.show()

## 7. Graphe 4 — Top GAB les plus exposés aux confrères 🎯C'est le cœur du reporting parc : quels automates sont le plus utilisés par les banques concurrentes(en % et en volume) ? Ce sont les GAB à surveiller / valoriser en priorité.

In [ ]:
gab_expo = (    df.groupby(['num_automate', 'type_porteur'])      .agg(nb_retrait=('nb_retrait', 'sum'))      .reset_index())gab_pivot = gab_expo.pivot(index='num_automate', columns='type_porteur', values='nb_retrait').fillna(0)gab_pivot['total_retraits'] = gab_pivot.get('Nos porteurs', 0) + gab_pivot.get('Confrères', 0)gab_pivot['pct_confreres']  = (gab_pivot.get('Confrères', 0) / gab_pivot['total_retraits'] * 100).round(1)gab_pivot = gab_pivot.reset_index()# On exclut les GAB avec très peu de volume (peu représentatifs) — seuil ajustableSEUIL_MIN_RETRAITS = 30gab_pivot_filtre = gab_pivot[gab_pivot['total_retraits'] >= SEUIL_MIN_RETRAITS]top_gab_expo_pct = gab_pivot_filtre.sort_values('pct_confreres', ascending=False).head(15)fig4 = px.bar(    top_gab_expo_pct,    x      = 'pct_confreres',    y      = 'num_automate',    orientation = 'h',    title  = f'🎯 Top 15 GAB les plus exposés aux confrères (% retraits confrères, min. {SEUIL_MIN_RETRAITS} retraits)',    labels = {'pct_confreres': '% retraits confrères', 'num_automate': 'GAB'},    text   = 'pct_confreres',    color_discrete_sequence = ['#e85d04'])fig4.update_traces(texttemplate='%{text:.1f}%', textposition='outside')fig4.update_layout(yaxis={'categoryorder': 'total ascending'})fig4.show()print(f"\nℹ️ {len(gab_pivot) - len(gab_pivot_filtre)} GAB exclus du classement (moins de {SEUIL_MIN_RETRAITS} retraits sur la période)")

### 7.1 Même classement, en volume brut de retraits confrères (pas en %)

In [ ]:
top_gab_expo_volume = gab_pivot.sort_values('Confrères', ascending=False).head(15) if 'Confrères' in gab_pivot.columns else pd.DataFrame()fig4b = px.bar(    top_gab_expo_volume,    x      = 'Confrères',    y      = 'num_automate',    orientation = 'h',    title  = '🎯 Top 15 GAB par volume de retraits confrères (nb retraits, 2025-2026)',    labels = {'Confrères': 'Nb retraits confrères', 'num_automate': 'GAB'},    text   = 'Confrères',    color_discrete_sequence = ['#e85d04'])fig4b.update_traces(texttemplate='%{text:,.0f}', textposition='outside')fig4b.update_layout(yaxis={'categoryorder': 'total ascending'})fig4b.show()

## 8. Graphe 5 — Montant moyen par retrait par banque (Top 10)

In [ ]:
moy = (    df.groupby(['label_banque', 'type_porteur'])      .agg(          nb_retrait    = ('nb_retrait',    'sum'),          montant_total = ('montant_total', 'sum')      )      .reset_index())moy['montant_moyen'] = moy['montant_total'] / moy['nb_retrait']moy = moy.sort_values('nb_retrait', ascending=False).head(10)fig5 = px.bar(    moy,    x      = 'label_banque',    y      = 'montant_moyen',    color  = 'type_porteur',    title  = '🏧 Parc GAB — Montant moyen par retrait — Top 10 banques (2025-2026)',    labels = {'montant_moyen': 'Montant moyen (€)', 'label_banque': 'Banque'},    color_discrete_map = {        'Nos porteurs' : '#003f88',        'Confrères'    : '#e85d04'    },    text_auto = '.0f')fig5.update_layout(xaxis_tickangle=-45)fig5.show()

## 9. Vision annuelle — Comparaison 2025 vs 2026 📆On change d'angle : au lieu du mois par mois, on regarde **année par année** pour voir si le poids des confrèresprogresse ou recule d'une année sur l'autre.⚠️ Si 2026 n'est pas terminée au moment de l'analyse, la comparaison brute (volumes totaux) favorisemécaniquement l'année qui a le plus de mois de données. Pense à vérifier le nombre de mois couverts par annéeci-dessous, et privilégie les **%** et les **moyennes mensuelles** plutôt que les volumes bruts si les deux annéesne sont pas comparables en nombre de mois.

In [ ]:
nb_mois_par_an = df.groupby('annee')['periode_tri'].nunique()print("Nombre de mois de données disponibles par année :")print(nb_mois_par_an)if nb_mois_par_an.nunique() > 1:    print("\n⚠️ Les années n'ont pas le même nombre de mois — privilégier les comparaisons en % ou en moyenne mensuelle ci-dessous.")else:    print("\n✅ Même nombre de mois pour chaque année — comparaison directe possible.")

### 9.1 KPIs annuels comparés

In [ ]:
kpi_annuel = (    df.groupby(['annee', 'type_porteur'])      .agg(          nb_retrait    = ('nb_retrait',    'sum'),          montant_total = ('montant_total', 'sum')      )      .reset_index())kpi_annuel_pivot = kpi_annuel.pivot(index='annee', columns='type_porteur', values='nb_retrait').fillna(0)kpi_annuel_pivot['total_retraits'] = kpi_annuel_pivot.get('Nos porteurs', 0) + kpi_annuel_pivot.get('Confrères', 0)kpi_annuel_pivot['pct_confreres']  = (kpi_annuel_pivot.get('Confrères', 0) / kpi_annuel_pivot['total_retraits'] * 100).round(1)kpi_annuel_pivot['nb_mois']        = nb_mois_par_ankpi_annuel_pivot['moy_mensuelle_confreres'] = (kpi_annuel_pivot.get('Confrères', 0) / kpi_annuel_pivot['nb_mois']).round(0)print("=" * 70)for annee in sorted(kpi_annuel_pivot.index):    row = kpi_annuel_pivot.loc[annee]    print(f"  📅 {int(annee)} ({int(row['nb_mois'])} mois de données)")    print(f"      Total retraits      : {row['total_retraits']:,.0f}")    print(f"      Confrères           : {row.get('Confrères', 0):,.0f}  ({row['pct_confreres']:.1f}%)")    print(f"      Moy. mensuelle confrères : {row['moy_mensuelle_confreres']:,.0f} / mois")    print("-" * 70)kpi_annuel_pivot

### 9.2 Graphe — Pie charts annuels côte à côte (Nos porteurs vs Confrères)

In [ ]:
annees_disponibles = sorted(df['annee'].dropna().unique())fig_annee_pie = make_subplots(    rows=1, cols=len(annees_disponibles),    specs=[[{'type': 'domain'}] * len(annees_disponibles)],    subplot_titles=[f"{int(a)}" for a in annees_disponibles])for i, annee in enumerate(annees_disponibles):    sous_df = df[df['annee'] == annee]    agg = sous_df.groupby('type_porteur')['nb_retrait'].sum().reset_index()    fig_annee_pie.add_trace(        go.Pie(            labels=agg['type_porteur'],            values=agg['nb_retrait'],            name=str(int(annee)),            marker=dict(colors=[                '#003f88' if t == 'Nos porteurs' else '#e85d04'                for t in agg['type_porteur']            ]),            hole=0.4,            textinfo='percent+label'        ),        row=1, col=i+1    )fig_annee_pie.update_layout(    title_text='🏧 Parc GAB — Nos Porteurs vs Confrères, par année',    showlegend=False)fig_annee_pie.show()

### 9.3 Graphe — Évolution YoY du % confrèresLa courbe la plus parlante du reporting : **le poids des confrères augmente-t-il ou diminue-t-il dans le temps ?**

In [ ]:
evol_pct_confreres = (    df.groupby('periode_tri')      .apply(lambda g: pd.Series({          'periode_label': g['periode_label'].iloc[0],          'annee': g['annee'].iloc[0],          'pct_confreres': g[g['type_porteur']=='Confrères']['nb_retrait'].sum() / g['nb_retrait'].sum() * 100      }))      .reset_index()      .sort_values('periode_tri'))ordre_periodes_pct = evol_pct_confreres['periode_label'].tolist()fig_pct_evol = px.line(    evol_pct_confreres,    x      = 'periode_label',    y      = 'pct_confreres',    color  = 'annee',    markers= True,    title  = '📈 Parc GAB — % de retraits confrères par mois (2025 vs 2026)',    labels = {'pct_confreres': '% retraits confrères', 'periode_label': 'Mois', 'annee': 'Année'},    category_orders = {'periode_label': ordre_periodes_pct})fig_pct_evol.update_xaxes(tickangle=-45)fig_pct_evol.update_yaxes(ticksuffix='%')fig_pct_evol.show()

### 9.4 Graphe — Comparaison mois-à-mois superposée (saisonnalité)Même mois (Jan, Fév, ...) sur l'axe X, une courbe par année : permet de voir si la saisonnalité se répète.

In [ ]:
mois_labels_ordre = ['Jan', 'Fév', 'Mar', 'Avr', 'Mai', 'Jun',                     'Jul', 'Aoû', 'Sep', 'Oct', 'Nov', 'Déc']evol_saison = (    df.groupby(['mois', 'annee'])      .agg(nb_retrait=('nb_retrait', 'sum'))      .reset_index())evol_saison['mois_nom'] = evol_saison['mois'].apply(lambda m: mois_labels_ordre[int(m)-1])evol_saison['annee'] = evol_saison['annee'].astype(int).astype(str)fig_saison = px.line(    evol_saison,    x      = 'mois_nom',    y      = 'nb_retrait',    color  = 'annee',    markers= True,    title  = '🔁 Parc GAB — Saisonnalité : retraits par mois, 2025 vs 2026 superposés',    labels = {'nb_retrait': 'Nb retraits (tous porteurs)', 'mois_nom': 'Mois', 'annee': 'Année'},    category_orders = {'mois_nom': mois_labels_ordre})fig_saison.show()

### 9.5 Graphe — Top banques confrères : qui progresse, qui recule (2025 vs 2026) ?

In [ ]:
top_banques_annee = (    df[df['type_porteur'] == 'Confrères']      .groupby(['label_banque', 'annee'])      .agg(nb_retrait=('nb_retrait', 'sum'))      .reset_index())# On garde le top 8 des banques confrères tous-temps confondus, pour ne pas surcharger le graphetop8_banques_confreres = (    top_banques_annee.groupby('label_banque')['nb_retrait'].sum()      .sort_values(ascending=False).head(8).index)top_banques_annee_filtre = top_banques_annee[top_banques_annee['label_banque'].isin(top8_banques_confreres)]top_banques_annee_filtre['annee'] = top_banques_annee_filtre['annee'].astype(int).astype(str)fig_banques_yoy = px.bar(    top_banques_annee_filtre,    x      = 'nb_retrait',    y      = 'label_banque',    color  = 'annee',    orientation = 'h',    barmode = 'group',    title  = '🏦 Top 8 banques confrères — Comparaison 2025 vs 2026',    labels = {'nb_retrait': 'Nb retraits', 'label_banque': 'Banque', 'annee': 'Année'},    text   = 'nb_retrait')fig_banques_yoy.update_traces(texttemplate='%{text:,.0f}', textposition='outside')fig_banques_yoy.update_layout(yaxis={'categoryorder': 'total ascending'})fig_banques_yoy.show()

## 10. Tableau récapitulatif final — Détail par GAB x Banque

In [ ]:
recap_gab = (    df.groupby(['num_automate', 'banque_affectation', 'lib_banque_affectation', 'type_porteur'])      .agg(          nb_retrait    = ('nb_retrait',    'sum'),          montant_total = ('montant_total', 'sum')      )      .reset_index())recap_gab['montant_moyen'] = (recap_gab['montant_total'] / recap_gab['nb_retrait']).round(2)# Part du retrait confrère/nos porteurs au sein de CHAQUE GABrecap_gab['part_au_sein_du_gab'] = (    recap_gab['nb_retrait'] / recap_gab.groupby('num_automate')['nb_retrait'].transform('sum') * 100).round(2)recap_gab = recap_gab.sort_values(['num_automate', 'nb_retrait'], ascending=[True, False])recap_gab.columns = [    'GAB', 'Code banque', 'Libellé banque', 'Type porteur',    'Nb retraits', 'Montant total (€)', 'Montant moyen (€)', 'Part au sein du GAB (%)']print(f"\n📋 Tableau récapitulatif détaillé — Parc GAB — 2025-2026")recap_gab

### 10.1 Tableau récapitulatif synthétique — Vue par GAB

In [ ]:
recap_synthese = gab_pivot[['num_automate', 'total_retraits', 'pct_confreres']].copy()if 'Nos porteurs' in gab_pivot.columns:    recap_synthese['nb_retrait_nos_porteurs'] = gab_pivot['Nos porteurs']if 'Confrères' in gab_pivot.columns:    recap_synthese['nb_retrait_confreres'] = gab_pivot['Confrères']recap_synthese = recap_synthese.sort_values('total_retraits', ascending=False)recap_synthese.columns = [c.replace('_', ' ').capitalize() for c in recap_synthese.columns]print(f"\n📋 Synthèse par GAB — {len(recap_synthese)} automates — 2025-2026")recap_synthese